In [1]:
from pathlib import Path
import pandas as pd
import geopandas as gpd

park_polygon_path = Path("park_polygon_clean.geojson")
tree_path = Path("tree_task_2.csv")
flower_path = Path("flower_task2.csv")

park_gdf = gpd.read_file(park_polygon_path)
tree_df = pd.read_csv(tree_path)
flower_df = pd.read_csv(flower_path)

print("Park polygon shape:", park_gdf.shape)
print("Park CRS:", park_gdf.crs)

print("Tree shape:", tree_df.shape)
print("Flower shape:", flower_df.shape)

Park polygon shape: (137, 16)
Park CRS: EPSG:3857
Tree shape: (274, 14)
Flower shape: (40, 14)


In [2]:
tree_feature = tree_df.copy()
tree_feature["feature_source"] = "tree"
tree_feature["feature_name"] = tree_feature["task_name"]
tree_feature["feature_category"] = "nature"

flower_feature = flower_df.copy()
flower_feature["feature_source"] = "flower"
flower_feature["feature_name"] = flower_feature["task_name"]
flower_feature["feature_category"] = "nature"

nature_feature_points = pd.concat(
    [tree_feature, flower_feature],
    ignore_index=True
)

print("Nature feature points:", nature_feature_points.shape)
print(nature_feature_points["feature_source"].value_counts())
print(nature_feature_points["feature_category"].value_counts())

Nature feature points: (314, 17)
feature_source
tree      274
flower     40
Name: count, dtype: int64
feature_category
nature    314
Name: count, dtype: int64


In [3]:
nature_feature_points["latitude"] = pd.to_numeric(
    nature_feature_points["latitude"],
    errors="coerce"
)

nature_feature_points["longitude"] = pd.to_numeric(
    nature_feature_points["longitude"],
    errors="coerce"
)

nature_gdf = gpd.GeoDataFrame(
    nature_feature_points,
    geometry=gpd.points_from_xy(
        nature_feature_points["longitude"],
        nature_feature_points["latitude"]
    ),
    crs="EPSG:4326"
)

# 转成和 park polygon 一样的 CRS
nature_gdf = nature_gdf.to_crs(park_gdf.crs)

print("Nature CRS after transform:", nature_gdf.crs)
print("Nature feature shape:", nature_gdf.shape)

Nature CRS after transform: EPSG:3857
Nature feature shape: (314, 18)


In [4]:
park_for_join = park_gdf[
    ["park_id", "park_name", "geometry"]
].copy()

nature_joined = gpd.sjoin(
    nature_gdf,
    park_for_join,
    how="inner",
    predicate="within"
)

print("Joined nature feature count:", nature_joined.shape)
print("Matched parks:", nature_joined["park_name"].nunique(dropna=True))

print(
    nature_joined[
        ["feature_source", "feature_category", "feature_name", "park_name"]
    ].head()
)

Joined nature feature count: (169, 21)
Matched parks: 39
  feature_source feature_category     feature_name                 park_name
0           tree           nature      English Elm              Fawkner Park
1           tree           nature        Dutch Elm  The Kings Domain and Tan
2           tree           nature  Moreton Bay Fig           Fitzroy Gardens
4           tree           nature  Moreton Bay Fig     Carlton Gardens South
5           tree           nature          Toromeo           Fitzroy Gardens


In [5]:
nature_joined_out = nature_joined[
    [
        "park_id",
        "park_name",
        "feature_source",
        "feature_name",
        "feature_category",
        "series_id",
        "latitude",
        "longitude",
    ]
].copy()

nature_joined_out = nature_joined_out.drop_duplicates(
    subset=[
        "park_id",
        "feature_source",
        "feature_name",
        "feature_category",
        "latitude",
        "longitude",
    ]
).copy()

output_path = Path("park_feature_join_tree_flower.csv")
nature_joined_out.to_csv(output_path, index=False, encoding="utf-8-sig")

print("Exported:", output_path)
print("Rows:", len(nature_joined_out))
print("Matched parks:", nature_joined_out["park_name"].nunique())

Exported: park_feature_join_tree_flower.csv
Rows: 169
Matched parks: 39


In [ ]:
nature_joined_out = nature_joined[
    [
        "park_id",
        "park_name",
        "feature_source",
        "feature_name",
        "feature_category",
        "series_id",
        "latitude",
        "longitude",
    ]
].copy()

nature_joined_out = nature_joined_out.drop_duplicates(
    subset=[
        "park_id",
        "feature_source",
        "feature_name",
        "feature_category",
        "latitude",
        "longitude",
    ]
).copy()

output_path = Path("park_feature_join_tree_flower.csv")
nature_joined_out.to_csv(output_path, index=False, encoding="utf-8-sig")

print("Exported:", output_path)
print("Rows:", len(nature_joined_out))
print("Matched parks:", nature_joined_out["park_name"].nunique())

,park_id,park_name,nature_feature_count
10,35,Fawkner Park,20
12,37,Flagstaff Gardens,16
6,14,Carlton Gardens North,12
11,36,Fitzroy Gardens,12
28,101,Royal Park,11
20,61,Kings Domain South,9
33,124,Treasury Gardens,9
7,15,Carlton Gardens South,9
36,128,Warun Biik Park,7
15,47,Grant Street Reserve,7
